In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix
from pandas.api.types import is_datetime64_any_dtype

In [3]:
model = joblib.load("/Users/manjakaranjatoson/Desktop/A_I/JEDHA/PROJECTS/CDSD/PPML/code/01_ETL/Bronze/00_Manja_test/output/joblib/model_retard_rf.joblib")
features_train = joblib.load("/Users/manjakaranjatoson/Desktop/A_I/JEDHA/PROJECTS/CDSD/PPML/code/01_ETL/Bronze/00_Manja_test/output/joblib/features_train.joblib")

In [4]:
df_new = pd.read_parquet("/Users/manjakaranjatoson/Desktop/A_I/JEDHA/PROJECTS/CDSD/PPML/data/processed/test_patrick/df_fe.parquet")
print(df_new.shape)
df_new.head()

(250295, 38)


,icao,type,codeshare_status,is_cargo,terminal_dep,terminal_arr,airline_name,aircraft_model,aircraft_family,num_seats,...,delay_minutes_clean,dep_hour,dep_dayofweek,dep_dayofmonth,dep_month,dep_hour_sin,dep_hour_cos,dep_dayofweek_sin,dep_dayofweek_cos,period_of_day
0,LFPG,departure,IsOperator,False,3,UNKNOWN,Nouvelair Tunisie,Airbus A320,Airbus A320 Family,165,...,150.0,20,5,4,10,-0.866025,0.500000,-0.974928,-0.222521,late_night
1,LFPG,departure,IsOperator,False,3,UNKNOWN,Nouvelair Tunisie,Airbus A320,Airbus A320 Family,165,...,140.0,20,5,4,10,-0.866025,0.500000,-0.974928,-0.222521,late_night
2,LFPG,departure,IsOperator,False,2E,2,Air France,Airbus A350-900,Airbus A350,325,...,54.0,21,5,4,10,-0.707107,0.707107,-0.974928,-0.222521,late_night
3,LFPG,departure,IsOperator,False,2E,3,Air France,Boeing 777-300,Boeing 777,355,...,33.0,21,5,4,10,-0.707107,0.707107,-0.974928,-0.222521,late_night
4,LFPG,departure,IsOperator,False,2E,1,Air France,Boeing 777,Boeing 777,355,...,19.0,21,5,4,10,-0.707107,0.707107,-0.974928,-0.222521,late_night


In [5]:
df_new.shape

(250295, 38)

In [ ]:
# print(df_new.iloc[0])
# df_new = df_new.iloc[1:].reset_index(drop=True)
# print("First line suppressed")

# df_new = df_new.sample(500, random_state=42).copy() # code for sampling if we got a bigger data

flight_date                             DATE_GENERATION
movement_date                       2026-04-09 02:34:54
flight_number                                       NaN
airline                                             NaN
airport_origin                                      NaN
                                           ...         
nombre_departs_destination                          NaN
nombre_arrivees_destination                         NaN
somme_depart_arrivee_destination                    NaN
congestion_destination                              NaN
retard arrivée                                      NaN
Name: 0, Length: 95, dtype: object
First line suppressed


In [6]:
cols_to_drop = [
    'cloud_base_dep', 'visibility_dep', 'cloud_base_arr', 'visibility_arr',
    'Vacances Scolaires', 'Label des Vacances', 'Label Jour Ferié',
    'LABEL_LYON', 'LABEL_TOULOUSE', 'LABEL_NICE', 'LABEL_MARSEILLE',
    'LABEL_CDG', 'LABEL_ORLY',
    "actual_arrival", "actual_departure", "actual_source_arrival",
    "actual_source_departure", "arrival_advance_min", "departure_delay_min",
    "departure_advance_min", "estimated_departure", "estimated_arrival",
    "arrival_delay_min"
]

df_new = df_new.drop(columns=cols_to_drop, errors="ignore")

In [7]:
target_col = "retard arrivée"

if target_col in df_new.columns:
    y_true = df_new[target_col].astype(int)
    X_new = df_new.drop(columns=[target_col], errors="ignore")
else:
    y_true = None
    X_new = df_new.copy()

In [14]:
print(y_true)

None


In [8]:
datetime_cols = [
    "flight_date",
    "movement_date",
    "scheduled_departure",
    "scheduled_arrival",
    "time_dep",
    "time_arr"
]

def add_datetime_features_safe(df, datetime_cols):
    df = df.copy()
    bad_datetime_cols = []

    for col in datetime_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

            if not is_datetime64_any_dtype(df[col]):
                bad_datetime_cols.append(col)

    usable_datetime_cols = [col for col in datetime_cols if col in df.columns and col not in bad_datetime_cols]

    if "flight_date" in usable_datetime_cols:
        df["flight_month"] = df["flight_date"].dt.month
        df["flight_day"] = df["flight_date"].dt.day
        df["flight_dayofweek"] = df["flight_date"].dt.dayofweek

    if "scheduled_departure" in usable_datetime_cols:
        df["sched_dep_hour"] = df["scheduled_departure"].dt.hour
        df["sched_dep_minute"] = df["scheduled_departure"].dt.minute

    if "scheduled_arrival" in usable_datetime_cols:
        df["sched_arr_hour"] = df["scheduled_arrival"].dt.hour
        df["sched_arr_minute"] = df["scheduled_arrival"].dt.minute

    print("Colonnes datetime problématiques droppées :", bad_datetime_cols)

    df = df.drop(columns=datetime_cols, errors="ignore")

    return df

X_new = add_datetime_features_safe(X_new, datetime_cols)

Colonnes datetime problématiques droppées : []


In [9]:
# Ajouter colonnes manquantes
for col in features_train:
    if col not in X_new.columns:
        X_new[col] = 0

# Garder uniquement les colonnes du train dans le bon ordre
X_new = X_new[features_train]

In [10]:
y_pred = model.predict(X_new)
y_proba = model.predict_proba(X_new)[:, 1]

In [12]:
df_result = df_new.copy()

df_result.insert(0, "prediction", y_pred)
df_result.insert(1, "proba_retard", y_proba)

In [13]:
df_result.to_csv("predictions_retard_vols.csv", index=False)
print("CSV exporté")

CSV exporté
